# CARLA state-policy training

Train the project's **8 → 24 → 16 → 3** driving policy from route-separated CARLA expert demonstrations. The notebook exports JSON weights that `neural_drive_agent.py` can load directly.

Policy contract version: `2` (generated from `policy_contract.py`).

## Goal

1. Load JSONL episodes recorded by `collect_expert_data.py`.
2. Keep complete routes in separate train, validation, and test sets.
3. Train steering, accelerator, and brake with supervised imitation learning.
4. Export a policy artifact for closed-loop CARLA evaluation.

> Real CARLA episodes are required by default. A synthetic smoke test only runs when `CARLA_SMOKE_TEST=1` is set explicitly, and its artifact is never deployable.

## Setup

In [ ]:
import importlib.util
import subprocess
import sys as _sys

REQUIRED_PACKAGES = ['numpy', 'torch', 'matplotlib']
missing_packages = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]
if missing_packages:
    print(f'Installing missing packages: {missing_packages}')
    subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All required packages already available; skipping installation.')

In [ ]:
import os
from pathlib import Path
import json, math, random, tempfile
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

SEED = 7
# GPU is opt-in: this MLP is tiny enough that host<->device transfer overhead
# usually outweighs any GPU speedup. Set ALLOW_GPU=1 to explicitly request CUDA.
ALLOW_GPU = os.environ.get('ALLOW_GPU', '0') == '1'
DEVICE = torch.device('cuda' if (ALLOW_GPU and torch.cuda.is_available()) else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
IN_COLAB = 'google.colab' in __import__('sys').modules
print({'device': str(DEVICE), 'torch': torch.__version__, 'in_colab': IN_COLAB,
       'policy_contract_version': 2})

### Data location

In Colab, place the `.jsonl` episode files in `MyDrive/CARLA/episodes`. The mount prompt is a Google authentication step and must be completed by you.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/CARLA/episodes')
    OUTPUT_DIR = Path('/content/drive/MyDrive/CARLA/models')
else:
    DATA_DIR = Path('episodes/expert')
    OUTPUT_DIR = Path('colab_artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'data_dir': str(DATA_DIR), 'output_dir': str(OUTPUT_DIR)})

## Steps

### 1. Normalize the eight live-policy inputs

`clamp`/`normalize_features` below are generated verbatim from `policy_contract.py` (the single source of truth shared with `neural_drive_agent.py`) so this notebook cannot silently drift from the deployed policy's input contract.

In [ ]:
POLICY_CONTRACT_VERSION = 2
POLICY_SIZES = [8, 24, 16, 3]
INPUT_NAMES = ('lane_error_m', 'heading_error_deg', 'speed_kmh', 'target_speed_kmh', 'front_obstacle_distance_m', 'red_light', 'curvature', 'stopped')

def clamp(value, low, high):
    return max(low, min(high, float(value)))

def normalize_features(features):
    """Normalize the eight live and training inputs identically."""
    if len(features) != len(INPUT_NAMES):
        raise ValueError(f"Expected {len(INPUT_NAMES)} policy inputs, got {len(features)}")
    lane, heading, speed, target, obstacle, red, curvature, stopped = features
    return [
        clamp(lane / 4.0, -1.0, 1.0),
        clamp(heading / 35.0, -1.0, 1.0),
        clamp(speed / 160.0, 0.0, 1.0),
        clamp(target / 160.0, 0.0, 1.0),
        clamp((obstacle - 40.0) / 40.0, -1.0, 1.0),
        clamp(red, 0.0, 1.0),
        clamp(curvature, -1.0, 1.0),
        clamp(stopped, 0.0, 1.0),
    ]

normalize_observation = normalize_features  # notebook-local alias

def teacher_action(values):
    """Rule-based expert used only to label synthetic smoke-test data."""
    lane, heading, speed, target, obstacle, red, curvature, stopped = values
    steering = clamp(-0.18 * lane - 0.035 * heading - 0.55 * curvature, -1, 1)
    desired = min(target, max(0.0, obstacle - 8.0) * 2.0)
    accelerator = clamp((desired - speed) / 25.0, 0, 1)
    brake = 1.0 if red > 0.7 or obstacle < 7.0 else clamp((speed - desired) / 12.0, 0, 1)
    return [steering, 0.0 if stopped else accelerator, brake]

### 2. Load and validate complete episodes

Real CARLA episodes are required by default. Set the environment variable `CARLA_SMOKE_TEST=1` to explicitly opt into a synthetic, non-deployable smoke test when no real data is available yet.

In [ ]:
MINIMUM_REAL_EPISODES = 10
MINIMUM_SPLIT_EPISODES = 6  # smallest episode count that can yield >=2/2/2 train/val/test

real_episode_files = sorted(DATA_DIR.rglob('*.jsonl')) if DATA_DIR.exists() else []
SMOKE_TEST_REQUESTED = os.environ.get('CARLA_SMOKE_TEST') == '1'

if real_episode_files:
    episode_files = real_episode_files
    SMOKE_TEST = False
    print(f'Using {len(episode_files)} real CARLA expert episode files.')
elif SMOKE_TEST_REQUESTED:
    SMOKE_TEST = True
    smoke_dir = Path(tempfile.gettempdir()) / 'carla_colab_smoke_episodes'
    smoke_dir.mkdir(parents=True, exist_ok=True)
    for old in smoke_dir.glob('*.jsonl'): old.unlink()
    rng = random.Random(SEED)
    for episode_index in range(9):
        path = smoke_dir / f'smoke-{episode_index:02d}.jsonl'
        with path.open('w', encoding='utf-8') as handle:
            for _ in range(240):
                obs = [rng.uniform(-4,4), rng.uniform(-35,35), rng.uniform(0,70),
                       rng.uniform(20,50), rng.uniform(3,80), rng.choice([0,0,0,1]),
                       rng.uniform(-0.5,0.5), rng.choice([0,0,1])]
                steer, accel, brake = teacher_action(obs)
                row = {'observation': obs, 'action': {'steering': steer,
                       'accelerator': accel, 'brake': brake},
                       'metadata': {'source': 'synthetic_smoke_test'}}
                handle.write(json.dumps(row) + '\n')
    episode_files = sorted(smoke_dir.glob('*.jsonl'))
    print('WARNING: CARLA_SMOKE_TEST=1 was set; using synthetic smoke-test data. '
          'The exported artifact will be explicitly marked non-deployable.')
else:
    raise RuntimeError(
        f'No .jsonl episodes found under {DATA_DIR}. Record real CARLA expert data with '
        "collect_expert_data.py, or set CARLA_SMOKE_TEST=1 to explicitly run a "
        'non-deployable synthetic smoke test instead.'
    )

episodes = {}
rejected_rows = 0
for path in episode_files:
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        try:
            row = json.loads(line)
            obs, action = row['observation'], row['action']
            metadata = row.get('metadata') or {}
            if not SMOKE_TEST and (metadata.get('route_conditioned') is not True or
                    metadata.get('policy_contract_version') != POLICY_CONTRACT_VERSION):
                raise ValueError
            steering = action.get('steering', action.get('steer'))
            if steering is None and 'steering_angle_deg' in action:
                steering = float(action['steering_angle_deg']) / 70.0
            target = [steering, action.get('accelerator', action.get('throttle')), action['brake']]
            if len(obs) != POLICY_SIZES[0]: raise ValueError
            values = list(map(float, obs)) + list(map(float, target))
            if not all(math.isfinite(v) for v in values): raise ValueError
            rows.append((normalize_observation(obs), target))
        except (KeyError, TypeError, ValueError, json.JSONDecodeError):
            rejected_rows += 1
    if rows: episodes[path.name] = rows

if SMOKE_TEST:
    assert len(episodes) >= MINIMUM_SPLIT_EPISODES, (
        f'Smoke test produced only {len(episodes)} non-empty episodes; '
        f'need at least {MINIMUM_SPLIT_EPISODES}.'
    )
else:
    assert len(episodes) >= MINIMUM_REAL_EPISODES, (
        f'Only {len(episodes)} non-empty real episodes were found under {DATA_DIR}; '
        f'at least {MINIMUM_REAL_EPISODES} are required before training.'
    )
print({'episodes': len(episodes), 'samples': sum(map(len, episodes.values())),
       'rejected_rows': rejected_rows, 'smoke_test': SMOKE_TEST})

### 3. Split by route, never by individual frame

In [ ]:
episode_names = sorted(episodes)
random.Random(SEED).shuffle(episode_names)
MIN_PER_SPLIT = 2
test_count = max(MIN_PER_SPLIT, round(len(episode_names) * 0.15))
validation_count = max(MIN_PER_SPLIT, round(len(episode_names) * 0.15))
test_names = episode_names[:test_count]
validation_names = episode_names[test_count:test_count + validation_count]
train_names = episode_names[test_count + validation_count:]
assert len(train_names) >= MIN_PER_SPLIT, (
    f'Only {len(train_names)} training episodes remain after the split; '
    f'need at least {MIN_PER_SPLIT}. Record more episodes.'
)
assert len(validation_names) >= MIN_PER_SPLIT and len(test_names) >= MIN_PER_SPLIT
assert set(train_names).isdisjoint(validation_names + test_names)
assert set(validation_names).isdisjoint(test_names)

def stack(names):
    pairs = [pair for name in names for pair in episodes[name]]
    x = torch.tensor([pair[0] for pair in pairs], dtype=torch.float32)
    y = torch.tensor([pair[1] for pair in pairs], dtype=torch.float32)
    return x, y

x_train, y_train = stack(train_names)
x_validation, y_validation = stack(validation_names)
x_test, y_test = stack(test_names)
# Move the static validation/test tensors to DEVICE once; the training loop and
# the final evaluation cell reuse these instead of re-transferring every epoch.
x_validation_device = x_validation.to(DEVICE)
y_validation_device = y_validation.to(DEVICE)
x_test_device = x_test.to(DEVICE)
y_test_device = y_test.to(DEVICE)
print({'train_episodes': train_names, 'validation_episodes': validation_names,
       'test_episodes': test_names, 'samples': [len(x_train), len(x_validation), len(x_test)]})

### 4. Train the MLP

In [ ]:
model = nn.Sequential(nn.Linear(8, 24), nn.Tanh(), nn.Linear(24, 16), nn.Tanh(), nn.Linear(16, 3)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()
loader = DataLoader(TensorDataset(x_train, y_train), batch_size=128, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED),
                    pin_memory=(DEVICE.type == 'cuda'))

MAX_EPOCHS = 80
PATIENCE = 12
MIN_IMPROVEMENT = 1e-5

history = []
best_validation = float('inf')
# Seed best_state with the model's own (finite, randomly-initialized) starting
# weights so a NaN-from-epoch-zero run never leaves best_state as None.
best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
epochs_without_improvement = 0
for epoch in range(MAX_EPOCHS):
    model.train(); train_total = 0.0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=(DEVICE.type == 'cuda'))
        batch_y = batch_y.to(DEVICE, non_blocking=(DEVICE.type == 'cuda'))
        optimizer.zero_grad(); prediction = model(batch_x)
        loss = loss_fn(prediction, batch_y); loss.backward(); optimizer.step()
        train_total += loss.item() * len(batch_x)
    train_loss = train_total / len(x_train)
    if not math.isfinite(train_loss):
        raise RuntimeError(f'Training loss became non-finite ({train_loss}) at epoch {epoch}.')
    model.eval()
    with torch.no_grad():
        validation_loss = loss_fn(model(x_validation_device), y_validation_device).item()
    if not math.isfinite(validation_loss):
        raise RuntimeError(f'Validation loss became non-finite ({validation_loss}) at epoch {epoch}.')
    history.append((train_loss, validation_loss))
    if validation_loss < best_validation - MIN_IMPROVEMENT:
        best_validation = validation_loss
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
            break
model.load_state_dict(best_state)
print({'best_validation_mse': best_validation, 'final_train_mse': history[-1][0],
       'epochs_run': len(history)})

### 5. Inspect learning and held-out errors

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot([v[0] for v in history], label='train')
plt.plot([v[1] for v in history], label='validation')
plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.title('Imitation-learning loss')
plt.legend(); plt.grid(alpha=.25); plt.show()

model.eval()
with torch.no_grad():
    test_prediction = model(x_test_device).cpu()
mae = torch.mean(torch.abs(test_prediction - y_test), dim=0).numpy()
test_mse = torch.mean((test_prediction - y_test) ** 2).item()
metrics = {'test_mse': float(test_mse), 'steering_mae': float(mae[0]),
           'accelerator_mae': float(mae[1]), 'brake_mae': float(mae[2])}
print(metrics)

### 6. Export weights compatible with the local driver

In [ ]:
linear_layers = [layer for layer in model if isinstance(layer, nn.Linear)]
artifact = {
    'sizes': POLICY_SIZES,
    'weights': [layer.weight.detach().cpu().tolist() for layer in linear_layers],
    'biases': [layer.bias.detach().cpu().tolist() for layer in linear_layers],
    'training_metadata': {
        'deployable': not SMOKE_TEST, 'source': 'carla_expert_jsonl' if not SMOKE_TEST else 'synthetic_smoke_test',
        'seed': SEED, 'policy_contract_version': POLICY_CONTRACT_VERSION,
        'route_conditioned': True,
        'train_episodes': train_names, 'validation_episodes': validation_names,
        'test_episodes': test_names, 'samples': int(len(x_train)+len(x_validation)+len(x_test)),
        'metrics': metrics,
    },
}
filename = 'nn_policy_smoke_test.json' if SMOKE_TEST else 'nn_policy_colab.json'
artifact_path = OUTPUT_DIR / filename
assert artifact_path.name != 'nn_policy.json', 'Refusing to overwrite the live nn_policy.json artifact.'
artifact_path.write_text(json.dumps(artifact, indent=2), encoding='utf-8')
assert all(np.isfinite(value) for layer in artifact['weights'] for row in layer for value in row)
print({'artifact': str(artifact_path), 'deployable': not SMOKE_TEST})

## Checks

- Train, validation, and test routes are disjoint, with at least two routes each.
- At least ten real episodes are required unless `CARLA_SMOKE_TEST=1` is set explicitly.
- Invalid rows are counted and excluded.
- Normalization is generated from `policy_contract.py`, not duplicated by hand.
- The JSON layer sizes match the local pure-Python loader.
- Synthetic smoke-test artifacts are explicitly marked non-deployable and are never written to `nn_policy.json`.

Offline errors are only a training check. A real policy must still pass closed-loop CARLA evaluation with the independent safety supervisor enabled.

## Next steps

1. Download `nn_policy_colab.json`.
2. Keep the existing `nn_policy.json` as a rollback copy.
3. Run the candidate on unseen CARLA routes with `--record-dir`.
4. Compare collisions, lane departures, completion, and safety interventions.
5. Promote the candidate only after closed-loop acceptance.